In [ ]:
!pip install pandas plotly

In [36]:
import pandas as pd
import plotly.express as px

In [37]:
def load_inset_lexicon(pos_filepath, neg_filepath):
    """
    Memuat file lexicon positif dan negatif dari InSet.
    """
    df_pos = pd.read_csv(pos_filepath, sep='\t')
    df_neg = pd.read_csv(neg_filepath, sep='\t')

    lexicon_dict = {}

    # Memetakan kata dan bobotnya (weight)
    for _, row in df_pos.iterrows():
        lexicon_dict[row['word']] = row['weight']

    for _, row in df_neg.iterrows():
        lexicon_dict[row['word']] = row['weight']

    return lexicon_dict

def analyze_sentiment_with_negation(comment_text, lexicon, comment_id):
    """
    Menghitung skor sentimen dengan algoritma negation handling.
    """
    # Menangani baris kosong
    if pd.isna(comment_text) or str(comment_text).strip() == "":
        return {
            'id_komentar': comment_id,
            'total_score': 0,
            'word_count': 0,
            'avg_score': 0.0,
            'label': 'netral'
        }

    words = str(comment_text).lower().split()
    total_score = 0
    word_count = len(words)

    # Daftar kata negasi umum dalam bahasa Indonesia
    negation_words = {'tidak', 'tak', 'bukan', 'belum', 'jangan', 'kurang', 'enggak', 'gak', 'ga', 'ndak'}

    for i, word in enumerate(words):
        if word in lexicon:
            word_score = lexicon[word]

            # Negation handling: Cek satu kata di belakang kata saat ini
            if i > 0 and words[i-1] in negation_words:
                # Membalikkan skor (positif jadi negatif, negatif jadi positif)
                word_score = word_score * -1

            total_score += word_score

    # Menghitung nilai rata-rata
    avg_score = total_score / word_count if word_count > 0 else 0

    # Klasifikasi Label (>= 0 Positif, < 0 Negatif)
    # Bisa ditambahkan kondisi == 0 untuk Netral jika dibutuhkan
    if total_score > 0:
        label = 'positif'
    elif total_score < 0:
        label = 'negatif'
    else:
        label = 'netral'

    return {
        'id_komentar': comment_id,
        'total_score': total_score,
        'word_count': word_count,
        'avg_score': round(avg_score, 4),
        'label': label
    }

def process_dataset(csv_file_path, lexicon):
    """
    Membaca dataset CSV dan memproses setiap barisnya.
    """
    # Membaca dataset CSV asli
    df = pd.read_csv(csv_file_path,sep=';')

    results = []

    for index, row in df.iterrows():
        # Kolom 'No' sebagai id_komentar, kolom 'text' sebagai teks komentar
        result = analyze_sentiment_with_negation(row['text'], lexicon, row['No'])
        results.append(result)

    # Mengembalikan hasil dalam bentuk DataFrame
    return pd.DataFrame(results)

In [38]:
def plot_interactive_sentiment(df, original_df=None):
    """
    Membuat grafik bar interaktif menggunakan Plotly.
    Jika original_df (dataset awal) disertakan, teks komentar bisa dimunculkan saat hover.
    """
    # [OPSIONAL] Menggabungkan teks komentar asli agar bisa dibaca saat di-hover
    # Jika Anda ingin menampilkan teks, pastikan id_komentar cocok dengan No/ID di data asli
    if original_df is not None:
        # Asumsi 'No' adalah ID dan 'text' adalah isi komentar di CSV asli Anda
        df = df.merge(original_df[['No', 'text']], left_on='id_komentar', right_on='No', how='left')

    # Pemetaan warna persis seperti contoh gambar Anda
    color_map = {
        'positif': '#1b9e77',
        'negatif': '#d95f02',
        'netral': '#8e8e8e'
    }

    # Membuat figure interaktif
    fig = px.bar(
        df,
        x=df.index, # Menggunakan index sebagai sumbu X agar rapat
        y='total_score',
        color='label',
        color_discrete_map=color_map,
        title='Sentiment Score per Document',
        labels={
            'index': 'Document',
            'total_score': 'Score',
            'label': 'Sentiment'
        },
        # Kolom apa saja yang ingin ditampilkan saat kursor diarahkan ke grafik
        hover_data=['id_komentar', 'word_count'] + (['text'] if original_df is not None else [])
    )

    # Styling agar tampilannya menyerupai ggplot (background abu-abu, grid putih)
    fig.update_layout(
        plot_bgcolor='#EAEAEA',
        paper_bgcolor='white',
        xaxis=dict(showgrid=True, gridcolor='white', tickmode='array', tickvals=df.index),
        yaxis=dict(showgrid=True, gridcolor='white'),
        legend_title_text='Sentiment',
        hovermode='closest' # Menampilkan tooltip hanya pada batang yang disorot
    )

    # Menyimpan hasil sebagai file HTML
    output_filename = 'interactive_sentiment_chart.html'
    fig.write_html(output_filename)
    print(f"Grafik interaktif berhasil disimpan sebagai '{output_filename}'.")
    print("Silakan buka file tersebut menggunakan browser (Chrome/Edge/Firefox/Safari).")

    # Menampilkan langsung jika dijalankan di environment seperti Jupyter/VS Code
    fig.show()

In [ ]:
# ==========================================
# EKSEKUSI PROGRAM (VERSI FINAL)
# ==========================================
if __name__ == "__main__":
    # 1. Load lexicon InSet (Pastikan file positive.tsv & negative.tsv ada di folder yang sama)
    print("Memuat lexicon...")
    lexicon = load_inset_lexicon('positive.tsv', 'negative.tsv')

    # 2. Proses dataset CSV Anda
    # Ganti 'data_komentar.csv' dengan nama file asli Anda.
    # Jika sebelumnya Anda menggunakan sep='\t' atau on_bad_lines='skip',
    # pastikan itu sudah disesuaikan di dalam fungsi process_dataset() di atas.
    nama_file_csv = 'komentar_bersih.csv'

    print(f"Memproses file {nama_file_csv}...")
    df_output = process_dataset(nama_file_csv, lexicon)

    # 3. Menampilkan ringkasan hasil (Menampilkan 5 baris pertama saja agar terminal tidak penuh)
    print("\n=== Ringkasan Hasil Analisis Sentimen ===")
    print(df_output.head())

    # Menampilkan total baris yang berhasil diproses
    print(f"\nTotal komentar yang diproses: {len(df_output)} baris")

    # 4. Menyimpan output ke file CSV baru
    nama_file_output = 'output_sentiment_score.csv'
    df_output.to_csv(nama_file_output, index=False)
    print(f"Hasil lengkap telah berhasil disimpan ke '{nama_file_output}'")

    df_asli = pd.read_csv('komentar_bersih.csv', sep=';') # Sesuaikan separator Anda
    df_output = process_dataset('komentar_bersih.csv', lexicon)
    #
    # Panggil fungsinya (tambahkan df_asli agar teks komentar muncul saat di-hover)
    plot_interactive_sentiment(df_output, df_asli)